In [0]:
%sql
-- ============================================================================
-- MPS II (Hunter) ONLY
-- Dx: E761 | Tx: Elaprase — NDC 54092070001, HCPCS J1743
-- Everything is TEMP VIEWs (including final output)
-- Dx Window: 2020-08-01 to 2025-11-30 | Tx Window: 2023-08-01 to 2025-11-30
-- ============================================================================

-- ----------------------------------------------------------------------------
-- 1) MPS II Treatment Table (5yr) - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
) t
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- ----------------------------------------------------------------------------
-- 2) MPS II 1Dx Specified (5yr) - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODES,
        KH_PLAN_ID AS KH_PLAN,
        PLACE_OF_SERVICE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        NULL AS PLACE_OF_SERVICE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
) combined
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- ----------------------------------------------------------------------------
-- 3) MPS II 2Dx Specified (patients) - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Specified AS
SELECT DISTINCT PATIENT_ID
FROM (
    SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS N
    FROM MPSII_1Dx_Specified
    GROUP BY PATIENT_ID
) x
WHERE N >= 2;


-- ----------------------------------------------------------------------------
-- 4) MPS II Tx Claims for 2Dx Specified - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Tx_Specified_Tx_Claims AS
SELECT *
FROM MPSII_TREATMENT_TABLE
WHERE PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM MPSII_2Dx_Specified);


-- ----------------------------------------------------------------------------
-- 5) MPS II Unspecified 1Dx (E763) - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Unspecified AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODES,
        KH_PLAN_ID AS KH_PLAN,
        PLACE_OF_SERVICE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        NULL AS PLACE_OF_SERVICE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
) combined
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- ----------------------------------------------------------------------------
-- 6) MPS II Unspecified 2Dx (patients) - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Unspecified AS
SELECT DISTINCT PATIENT_ID
FROM (
    SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS N
    FROM MPSII_1Dx_Unspecified
    GROUP BY PATIENT_ID
) x
WHERE N >= 2;


-- ----------------------------------------------------------------------------
-- 7) MPS II Incremental Patients:
--    (E763 2Dx) AND Elaprase-treated AND NOT in specified 2Dx
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_Incremental_Patients AS
SELECT DISTINCT PATIENT_ID
FROM MPSII_2Dx_Unspecified
WHERE PATIENT_ID IN (
    SELECT DISTINCT PATIENT_ID
    FROM MPSII_TREATMENT_TABLE
    WHERE CODE IN ('54092070001','540920700','J1743')
)
AND PATIENT_ID NOT IN (
    SELECT DISTINCT PATIENT_ID
    FROM MPSII_2Dx_Specified
);


-- ----------------------------------------------------------------------------
-- 8) MPS II All Dx with specialty - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_All_Dx AS
SELECT
    a.*,
    b.HCO_PRIMARY_NPI,
    b.PRIMARY_SPECIALTY,
    b.SECONDARY_SPECIALTY,
    CASE
        WHEN primary_specialty LIKE '%Genetic%' OR secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty LIKE '%Psychiatry & Neurology%'
          OR secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
          OR primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty LIKE '%Nurse Practitioner%' OR primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty LIKE '%Internal Medicine%' OR secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty LIKE '%Family Medicine%'  OR secondary_specialty LIKE '%Family Medicine%'  THEN 'PCP'
        WHEN a.NPI IS NULL THEN 'NA'
        ELSE 'Others'
    END AS SPECIALTY
FROM (
    SELECT *
    FROM MPSII_1Dx_Specified
    WHERE PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM MPSII_2Dx_Specified)

    UNION

    SELECT *
    FROM MPSII_1Dx_Unspecified
    WHERE PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM MPSII_Incremental_Patients)
) a
LEFT JOIN com_edp_prd.com_raw.kom_providers b
  ON a.NPI = b.NPI;


-- ----------------------------------------------------------------------------
-- 9) MPS II Treatment Table (2yr, 2023-08-01 to 2025-11-30) - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE_2023_to_25 AS
SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- ----------------------------------------------------------------------------
-- 10) MPS II All Tx (2yr) with specialty - temp view
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_All_Tx_2023_to_25 AS
SELECT
    a.*,
    b.HCO_PRIMARY_NPI,
    b.PRIMARY_SPECIALTY,
    b.SECONDARY_SPECIALTY,
    CASE
        WHEN primary_specialty LIKE '%Genetic%' OR secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty LIKE '%Psychiatry & Neurology%'
          OR secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
          OR primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty LIKE '%Nurse Practitioner%' OR primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty LIKE '%Internal Medicine%' OR secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty LIKE '%Family Medicine%'  OR secondary_specialty LIKE '%Family Medicine%'  THEN 'PCP'
        WHEN a.NPI IS NULL THEN 'NA'
        ELSE 'Others'
    END AS SPECIALTY
FROM (
    -- specified 2Dx patients
    SELECT *
    FROM MPSII_TREATMENT_TABLE_2023_to_25
    WHERE PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM MPSII_2Dx_Specified)

    UNION

    -- incremental patients, but only Elaprase-related codes
    SELECT *
    FROM MPSII_TREATMENT_TABLE_2023_to_25
    WHERE PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM MPSII_Incremental_Patients)
      AND CODE IN ('54092070001','540920700','J1743')
) a
LEFT JOIN com_edp_prd.com_raw.kom_providers b
  ON a.NPI = b.NPI;


-- ============================================================================
-- FINAL OUTPUT (TEMP VIEW): Consolidated HCO-level summary for MPS II only
-- ============================================================================
CREATE OR REPLACE TEMP VIEW MPSII_HCO_SUMMARY AS
WITH
hco_mpsii_dx AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT PATIENT_ID) AS MPSII_DX
    FROM MPSII_All_Dx
    WHERE HCO_PRIMARY_NPI IS NOT NULL
    GROUP BY 1
),
hco_mpsii_tx AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT PATIENT_ID) AS MPSII_TX
    FROM MPSII_All_Tx_2023_to_25
    WHERE HCO_PRIMARY_NPI IS NOT NULL
    GROUP BY 1
),
all_hcos AS (
    SELECT HCO_PRIMARY_NPI FROM hco_mpsii_dx
    UNION
    SELECT HCO_PRIMARY_NPI FROM hco_mpsii_tx
)
SELECT
    h.HCO_PRIMARY_NPI,
    COALESCE(d.MPSII_DX, 0) AS MPSII_DX,
    COALESCE(t.MPSII_TX, 0) AS MPSII_TX
FROM all_hcos h
LEFT JOIN hco_mpsii_dx d ON h.HCO_PRIMARY_NPI = d.HCO_PRIMARY_NPI
LEFT JOIN hco_mpsii_tx t ON h.HCO_PRIMARY_NPI = t.HCO_PRIMARY_NPI
ORDER BY (COALESCE(d.MPSII_DX,0) + COALESCE(t.MPSII_TX,0)) DESC;



SELECT * FROM MPSII_HCO_SUMMARY;


In [0]:
com_edp_prd.cmpa_insights_internal_schema.reference_file_0109

In [0]:
%sql
-- Bring MPSII metrics (MPSII_DX, MPSII_TX) onto the provider attributes output

WITH t1 AS (
  SELECT * FROM MPSII_HCO_SUMMARY
),
prov AS (
  SELECT DISTINCT
      NPI,
      ORGANIZATION_NAME,
      PROVIDER_ADDRESS,
      PROVIDER_ZIP,
      PROVIDER_TYPE,
      PROVIDER_STATE,
      PROVIDER_CITY
  FROM com_edp_prd.com_raw.kom_providers
  WHERE NPI IN (SELECT DISTINCT HCO_PRIMARY_NPI FROM t1)
)
SELECT
    p.NPI,
    p.ORGANIZATION_NAME,
    p.PROVIDER_ADDRESS,
    p.PROVIDER_CITY,
    p.PROVIDER_STATE,
    p.PROVIDER_ZIP,
    p.PROVIDER_TYPE,
    t1.MPSII_DX,
    t1.MPSII_TX
FROM prov p
LEFT JOIN t1
  ON p.NPI = t1.HCO_PRIMARY_NPI
ORDER BY (COALESCE(t1.MPSII_DX,0) + COALESCE(t1.MPSII_TX,0)) DESC;


In [0]:
%sql
WITH dx AS (
    SELECT DISTINCT HCO_PRIMARY_NPI, patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Dx
    WHERE HCO_PRIMARY_NPI IS NOT NULL
),
tx AS (
    SELECT DISTINCT HCO_PRIMARY_NPI, patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Tx_2023_to_25
    WHERE HCO_PRIMARY_NPI IS NOT NULL
),

dx_count AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_DX
    FROM dx
    GROUP BY 1
),
tx_count AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_TX
    FROM tx
    GROUP BY 1
),

dx_tx_union AS (
    SELECT HCO_PRIMARY_NPI, patient_id FROM dx
    UNION
    SELECT HCO_PRIMARY_NPI, patient_id FROM tx
),

total_distinct_patients AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_TOTAL_DISTINCT_PATIENTS
    FROM dx_tx_union
    GROUP BY 1
),

all_hcos AS (
    SELECT HCO_PRIMARY_NPI FROM dx_count
    UNION
    SELECT HCO_PRIMARY_NPI FROM tx_count
)

SELECT
    h.HCO_PRIMARY_NPI,
    COALESCE(d.MPSII_DX, 0) AS MPSII_DX,
    COALESCE(t.MPSII_TX, 0) AS MPSII_TX,
    COALESCE(p.MPSII_TOTAL_DISTINCT_PATIENTS, 0) AS MPSII_TOTAL_DISTINCT_PATIENTS,
    (COALESCE(d.MPSII_DX,0) + COALESCE(t.MPSII_TX,0)) AS MPSII_DX_TX_SUM
FROM all_hcos h
LEFT JOIN dx_count d ON h.HCO_PRIMARY_NPI = d.HCO_PRIMARY_NPI
LEFT JOIN tx_count t ON h.HCO_PRIMARY_NPI = t.HCO_PRIMARY_NPI
LEFT JOIN total_distinct_patients p ON h.HCO_PRIMARY_NPI = p.HCO_PRIMARY_NPI
ORDER BY MPSII_TOTAL_DISTINCT_PATIENTS DESC;


In [0]:
%sql
WITH t1 AS (
    SELECT
        h.HCO_PRIMARY_NPI,
        COALESCE(d.MPSII_DX, 0) AS MPSII_DX,
        COALESCE(t.MPSII_TX, 0) AS MPSII_TX,
        (COALESCE(d.MPSII_DX, 0) + COALESCE(t.MPSII_TX, 0)) AS MPSII_TOTAL
    FROM (
        SELECT HCO_PRIMARY_NPI FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Dx
        UNION
        SELECT HCO_PRIMARY_NPI FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Tx_2023_to_25
    ) h
    LEFT JOIN (
        SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_DX
        FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Dx
        WHERE HCO_PRIMARY_NPI IS NOT NULL
        GROUP BY 1
    ) d ON h.HCO_PRIMARY_NPI = d.HCO_PRIMARY_NPI
    LEFT JOIN (
        SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_TX
        FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Tx_2023_to_25
        WHERE HCO_PRIMARY_NPI IS NOT NULL
        GROUP BY 1
    ) t ON h.HCO_PRIMARY_NPI = t.HCO_PRIMARY_NPI
)

SELECT DISTINCT
    p.NPI AS HCO_PRIMARY_NPI,
    p.ORGANIZATION_NAME,
    p.PROVIDER_TYPE,
    p.PROVIDER_ADDRESS,
    p.PROVIDER_CITY,
    p.PROVIDER_STATE,
    p.PROVIDER_ZIP,
    t1.MPSII_DX,
    t1.MPSII_TX,
    t1.MPSII_TOTAL
FROM t1
LEFT JOIN com_edp_prd.com_raw.kom_providers p
    ON t1.HCO_PRIMARY_NPI = p.NPI
WHERE p.NPI IS NOT NULL
ORDER BY t1.MPSII_TOTAL DESC;


In [0]:
%sql
WITH tier1_npi_list AS (
    SELECT *
    FROM VALUES
    ('1437365186', 1),
    ('1235234535', 1),
    ('1295789907', 1),
    ('1114969169', 1),
    ('1346297843', 1),
    ('1336245828', 1),
    ('1104001858', 1),
    ('1104819366', 1),
    ('1811080526', 1),
    ('1295820256', 1),
    ('1366515488', 1),
    ('1912939703', 1),
    ('1205935012', 1),
    ('1750482022', 1),
    ('1194787218', 1),
    ('1548212988', 1),
    ('1891765178', 1),
    ('1023105400', 1),
    ('1649347469', 1),
    ('1336495910', 1),
    ('1760476659', 1),
    ('1235339227', 1),
    ('1184649345', 1),
    ('1578693321', 1),
    ('1013143213', 1),
    ('1013062769', 1),
    ('1518911338', 1),
    ('1073053757', 1),
    ('1467442749', 1),
    ('1144548322', 1),
    ('1285174649', 1),
    ('1659877280', 1),
    ('1003063280', 1),
    ('1275564098', 1),
    ('1093894131', 1),
    ('1013924372', 1),
    ('1609824010', 1),
    ('1669429577', 1),
    ('1033439732', 1),
    ('1760480503', 1),
    ('1235148594', 1),
    ('1750458485', 1),
    ('1114924834', 1),
    ('1083789630', 1),
    ('1669462420', 1),
    ('1477643690', 1),
    ('1649261462', 1),
    ('1023188851', 1),
    ('1043447253', 1),
    ('1477549756', 1),
    ('1154302727', 1),
    ('1083949382', 1),
    ('1003961251', 1),
    ('1235214834', 1),
    ('1265694442', 1),
    ('1376544320', 1),
    ('1679973364', 1),
    ('1568596765', 1),
    ('1003878539', 1),
    ('1326092404', 1),
    ('1093808040', 1),
    ('1598784555', 1),
    ('1689747552', 1),
    ('1164426896', 1),
    ('1063702785', 1),
    ('1083630073', 1),
    ('1235582925', 1),
    ('1639370059', 1),
    ('1275694184', 1),
    ('1255577466', 1),
    ('1669683512', 1),
    ('1932280666', 1),
    ('1013924182', 1),
    ('1225249865', 1)
    AS t(npi, tier)
),

dx AS (
    SELECT DISTINCT HCO_PRIMARY_NPI, patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Dx
    WHERE HCO_PRIMARY_NPI IS NOT NULL
),

tx AS (
    SELECT DISTINCT HCO_PRIMARY_NPI, patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Tx_2023_to_25
    WHERE HCO_PRIMARY_NPI IS NOT NULL
),

dx_count AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_DX
    FROM dx
    GROUP BY 1
),

tx_count AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_TX
    FROM tx
    GROUP BY 1
),

dx_tx_union AS (
    SELECT HCO_PRIMARY_NPI, patient_id FROM dx
    UNION
    SELECT HCO_PRIMARY_NPI, patient_id FROM tx
),

total_distinct_patients AS (
    SELECT HCO_PRIMARY_NPI,
           COUNT(DISTINCT patient_id) AS MPSII_TOTAL_DISTINCT_PATIENTS
    FROM dx_tx_union
    GROUP BY 1
),

all_hcos AS (
    SELECT HCO_PRIMARY_NPI FROM dx_count
    UNION
    SELECT HCO_PRIMARY_NPI FROM tx_count
)

SELECT DISTINCT
    h.HCO_PRIMARY_NPI,
    p.ORGANIZATION_NAME,
    p.PROVIDER_ADDRESS,
    p.PROVIDER_CITY,
    p.PROVIDER_STATE,
    p.PROVIDER_ZIP,
    COALESCE(d.MPSII_DX, 0) AS MPSII_DX,
    COALESCE(t.MPSII_TX, 0) AS MPSII_TX,
    COALESCE(tp.MPSII_TOTAL_DISTINCT_PATIENTS, 0) AS MPSII_TOTAL_DISTINCT_PATIENTS,

    CASE 
        WHEN tier.npi IS NOT NULL THEN 1
        ELSE 2
    END AS TIER_FLAG

FROM all_hcos h
LEFT JOIN dx_count d ON h.HCO_PRIMARY_NPI = d.HCO_PRIMARY_NPI
LEFT JOIN tx_count t ON h.HCO_PRIMARY_NPI = t.HCO_PRIMARY_NPI
LEFT JOIN total_distinct_patients tp ON h.HCO_PRIMARY_NPI = tp.HCO_PRIMARY_NPI
LEFT JOIN com_edp_prd.com_raw.kom_providers p ON h.HCO_PRIMARY_NPI = p.NPI
LEFT JOIN tier1_npi_list tier ON h.HCO_PRIMARY_NPI = tier.npi

ORDER BY MPSII_TOTAL_DISTINCT_PATIENTS DESC;


In [0]:
%sql
WITH tier1_npi_list AS (
    SELECT *
    FROM VALUES
    ('1437365186'),
    ('1235234535'),
    ('1295789907'),
    ('1114969169'),
    ('1346297843'),
    ('1336245828'),
    ('1104001858'),
    ('1104819366'),
    ('1811080526'),
    ('1295820256'),
    ('1366515488'),
    ('1912939703'),
    ('1205935012'),
    ('1750482022'),
    ('1194787218'),
    ('1548212988'),
    ('1891765178'),
    ('1023105400'),
    ('1649347469'),
    ('1336495910'),
    ('1760476659'),
    ('1235339227'),
    ('1184649345'),
    ('1578693321'),
    ('1013143213'),
    ('1013062769'),
    ('1518911338'),
    ('1073053757'),
    ('1467442749'),
    ('1144548322'),
    ('1285174649'),
    ('1659877280'),
    ('1003063280'),
    ('1275564098'),
    ('1093894131'),
    ('1013924372'),
    ('1609824010'),
    ('1669429577'),
    ('1033439732'),
    ('1760480503'),
    ('1235148594'),
    ('1750458485'),
    ('1114924834'),
    ('1083789630'),
    ('1669462420'),
    ('1477643690'),
    ('1649261462'),
    ('1023188851'),
    ('1043447253'),
    ('1477549756'),
    ('1154302727'),
    ('1083949382'),
    ('1003961251'),
    ('1235214834'),
    ('1265694442'),
    ('1376544320'),
    ('1679973364'),
    ('1568596765'),
    ('1003878539'),
    ('1326092404'),
    ('1093808040'),
    ('1598784555'),
    ('1689747552'),
    ('1164426896'),
    ('1063702785'),
    ('1083630073'),
    ('1235582925'),
    ('1639370059'),
    ('1275694184'),
    ('1255577466'),
    ('1669683512'),
    ('1932280666'),
    ('1013924182'),
    ('1225249865')
    AS t(npi)
),

dx AS (
    SELECT DISTINCT HCO_PRIMARY_NPI, patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Dx
    WHERE HCO_PRIMARY_NPI IS NOT NULL
),

tx AS (
    SELECT DISTINCT HCO_PRIMARY_NPI, patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_All_Tx_2023_to_25
    WHERE HCO_PRIMARY_NPI IS NOT NULL
),

dx_count AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_DX
    FROM dx
    GROUP BY 1
),

tx_count AS (
    SELECT HCO_PRIMARY_NPI, COUNT(DISTINCT patient_id) AS MPSII_TX
    FROM tx
    GROUP BY 1
),

dx_tx_union AS (
    SELECT HCO_PRIMARY_NPI, patient_id FROM dx
    UNION
    SELECT HCO_PRIMARY_NPI, patient_id FROM tx
),

total_distinct_patients AS (
    SELECT 
        HCO_PRIMARY_NPI,
        COUNT(DISTINCT patient_id) AS MPSII_TOTAL
    FROM dx_tx_union
    GROUP BY 1
),

hcos AS (
    SELECT DISTINCT HCO_PRIMARY_NPI
    FROM dx_tx_union
),

provider_dedup AS (
    SELECT *
    FROM (
        SELECT 
            NPI,
            ORGANIZATION_NAME,
            PROVIDER_TYPE,
            PROVIDER_ADDRESS,
            PROVIDER_CITY,
            PROVIDER_STATE,
            PROVIDER_ZIP,
            ROW_NUMBER() OVER (PARTITION BY NPI ORDER BY ORGANIZATION_NAME) AS rn
        FROM com_edp_prd.com_raw.kom_providers
    )
    WHERE rn = 1
)

SELECT
    h.HCO_PRIMARY_NPI,
    p.ORGANIZATION_NAME,
    p.PROVIDER_TYPE,
    p.PROVIDER_ADDRESS,
    p.PROVIDER_CITY,
    p.PROVIDER_STATE,
    p.PROVIDER_ZIP,
    COALESCE(d.MPSII_DX, 0) AS MPSII_DX,
    COALESCE(t.MPSII_TX, 0) AS MPSII_TX,
    COALESCE(tp.MPSII_TOTAL, 0) AS MPSII_TOTAL,

    CASE 
        WHEN tier.npi IS NOT NULL THEN 1
        ELSE 2
    END AS NPI_TIER

FROM hcos h
LEFT JOIN dx_count d ON h.HCO_PRIMARY_NPI = d.HCO_PRIMARY_NPI
LEFT JOIN tx_count t ON h.HCO_PRIMARY_NPI = t.HCO_PRIMARY_NPI
LEFT JOIN total_distinct_patients tp ON h.HCO_PRIMARY_NPI = tp.HCO_PRIMARY_NPI
LEFT JOIN provider_dedup p ON h.HCO_PRIMARY_NPI = p.NPI
LEFT JOIN tier1_npi_list tier ON h.HCO_PRIMARY_NPI = tier.npi
-- WHERE p.NPI IS NOT NULL
ORDER BY MPSII_TOTAL DESC;
